# Graph Message Passing Practice

**Topic:** Graph Neural Networks &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `A` `(400, 400)`: symmetric adjacency matrix of an undirected graph with 4 hidden communities (stochastic block model: dense within, sparse between).
- `X` `(400, 8)`: node features with weak community signal.
- `train_idx`, `y_train`: labels for 200 nodes. `test_idx`: the 200 nodes you must classify (`y_test` is used by the evaluation cell).

## Your task

1. Normalize the adjacency: `Â = D⁻¹(A + I)` (row-normalized, self-loops added).
2. Propagate: `H = Â @ (Â @ X)`, two hops of neighborhood averaging. (Optionally concatenate `X` and `H`.)
3. Train any classifier on `H[train_idx]` (the solution uses scikit-learn's `LogisticRegression`, which the site editor loads on import and which needs `pip install scikit-learn` on your own machine) and set `predictions` for the nodes in `test_idx` (in that order).

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

rng = np.random.default_rng(531)
n, k = 400, 4
communities = np.repeat(np.arange(k), n // k)
rng.shuffle(communities)

P_IN, P_OUT = 0.10, 0.01
same = communities[:, None] == communities[None, :]
probs = np.where(same, P_IN, P_OUT)
upper = np.triu(rng.random((n, n)) < probs, k=1)
A = (upper | upper.T).astype(float)

centers = rng.normal(0, 1.0, (k, 8))
X = centers[communities] + rng.normal(0, 2.2, (n, 8))   # weak node signal

perm = rng.permutation(n)
train_idx, test_idx = perm[:200], perm[200:]
y_train = communities[train_idx]
y_test = communities[test_idx]
print(f"graph: {n} nodes, {int(A.sum() // 2)} edges, avg degree {A.sum(1).mean():.1f}")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.82

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {_cls!r:>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet. Revisit your model and try again.")


## Solution

Row-normalize the self-looped adjacency and average features over the neighborhood twice, two hops of message passing smooth node features toward their community means. Logistic regression on `[X, H]` then separates the four communities.

```python
import numpy as np
from sklearn.linear_model import LogisticRegression

A_hat = A + np.eye(len(A))
A_hat = A_hat / A_hat.sum(axis=1, keepdims=True)   # D^-1 (A + I)

H = A_hat @ (A_hat @ X)                            # two hops of propagation
features = np.hstack([X, H])

clf = LogisticRegression(max_iter=3000)
clf.fit(features[train_idx], y_train)
predictions = clf.predict(features[test_idx])
```
